# FinBERT-GF — Inference Pipeline (no retraining)
Rebuilds `lm_scaler` / `temporal_scaler` deterministically (same `random_state=42` split as training), loads `best_model.pt`, and serves tone classification over a Flask API for the Trading Floor simulator.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, gc, re, random, warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import requests
from io import StringIO

from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)


## Rebuild dataset + labels (identical to training notebook)

In [ ]:
# --- Bond yields (ECB SDW) ---
url = ("https://data-api.ecb.europa.eu/service/data/YC/"
       "B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y"
       "?format=csvdata&startPeriod=1999-01-01")
r = requests.get(url)
lines = r.text.splitlines()
for i, line in enumerate(lines):
    if "TIME_PERIOD" in line:
        header = i
        break
bond = pd.read_csv(StringIO(r.text), skiprows=header)
bond = bond[['TIME_PERIOD', 'OBS_VALUE']]
bond.columns = ['date', 'Bond_Yield']
bond['date'] = pd.to_datetime(bond['date'])
bond['Bond_Yield'] = pd.to_numeric(bond['Bond_Yield'], errors='coerce')
bond = bond.dropna().sort_values('date')
print(f"Bond rows: {len(bond):,}")

# --- Speeches ---
speeches = pd.read_csv('/content/drive/MyDrive/ECB/all_ECB_speeches (1).csv', sep='|')
speeches['date'] = pd.to_datetime(speeches['date'], errors='coerce')
speeches = speeches.dropna(subset=['date'])
speeches['contents'] = speeches['contents'].astype(str).fillna('')
speeches_daily = speeches.groupby('date')['contents'].apply(' '.join).reset_index()
print(f"Daily speeches: {len(speeches_daily):,}")

# --- Merge + temporal features + labels ---
bond['yield_change'] = bond['Bond_Yield'].diff()

df = pd.merge_asof(
    speeches_daily.sort_values('date'),
    bond.sort_values('date'),
    on='date', direction='backward', tolerance=pd.Timedelta('7D')
).sort_values('date').reset_index(drop=True)

df['yield_change'] = df['Bond_Yield'].diff()
df['volatility']   = df['Bond_Yield'].rolling(5).std()
df['delta_next']   = df['Bond_Yield'].shift(-2) - df['Bond_Yield']

upper_thr = df['delta_next'].quantile(0.65)
lower_thr = df['delta_next'].quantile(0.35)

def label(x):
    if x > upper_thr: return 'UP'
    if x < lower_thr: return 'DOWN'
    return 'NEUTRAL'

label_map = {'DOWN': 0, 'NEUTRAL': 1, 'UP': 2}  # 0=Dovish, 1=Neutral, 2=Hawkish
df['label_next'] = df['delta_next'].apply(label).map(label_map)
df = df.dropna(subset=['label_next']).reset_index(drop=True)
print("Labels rebuilt:", df['label_next'].value_counts().to_dict())


In [ ]:
# --- LM (Loughran-McDonald) features ---
LM_DICT_PATH = "/content/drive/MyDrive/ECB/Loughran-McDonald_MasterDictionary_1993-2024.csv"
lm_df = pd.read_csv(LM_DICT_PATH,
                    usecols=['Word','Negative','Positive','Uncertainty','Strong_Modal','Weak_Modal'])

lm_negative     = set(lm_df[lm_df['Negative']     > 0]['Word'].str.upper())
lm_positive     = set(lm_df[lm_df['Positive']     > 0]['Word'].str.upper())
lm_uncertainty  = set(lm_df[lm_df['Uncertainty']  > 0]['Word'].str.upper())
lm_strong_modal = set(lm_df[lm_df['Strong_Modal'] > 0]['Word'].str.upper())
lm_weak_modal   = set(lm_df[lm_df['Weak_Modal']   > 0]['Word'].str.upper())
del lm_df; gc.collect()

def compute_lm_features(text):
    words = re.findall(r'[A-Z]+', text.upper())
    n = max(len(words), 1)
    neg  = sum(1 for w in words if w in lm_negative)     / n * 100
    pos  = sum(1 for w in words if w in lm_positive)     / n * 100
    unc  = sum(1 for w in words if w in lm_uncertainty)  / n * 100
    strn = sum(1 for w in words if w in lm_strong_modal) / n * 100
    weak = sum(1 for w in words if w in lm_weak_modal)   / n * 100
    return {
        'lm_negative': neg, 'lm_positive': pos, 'lm_uncertainty': unc,
        'lm_net_sentiment': pos - neg, 'lm_modal_certainty': strn - weak,
    }

lm_feat_df = pd.DataFrame(df['contents'].astype(str).apply(compute_lm_features).tolist())
df = pd.concat([df, lm_feat_df], axis=1)
print("LM features rebuilt")


In [ ]:
# --- Same split + scalers as training (random_state=42 => identical split) ---
labels = df['label_next'].values.astype(int)

train_idx, temp_idx = train_test_split(
    np.arange(len(df)), test_size=0.30, stratify=labels, random_state=42)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, stratify=labels[temp_idx], random_state=42)

TEMPORAL_COLS = ['yield_change', 'volatility', 'Bond_Yield']
df[TEMPORAL_COLS] = df[TEMPORAL_COLS].fillna(0)
temporal_scaler = StandardScaler().fit(df.iloc[train_idx][TEMPORAL_COLS].values.astype(np.float32))

LM_FEATURE_COLS = ['lm_negative','lm_positive','lm_uncertainty','lm_net_sentiment','lm_modal_certainty']
lm_scaler = StandardScaler().fit(df.iloc[train_idx][LM_FEATURE_COLS].values.astype(np.float32))

print("Scalers rebuilt — ready for inference")
print("Latest known market row (for live proxy use):")
print(df[['date'] + TEMPORAL_COLS].tail(1))


## Model architecture (must match training exactly) + load checkpoint

In [ ]:
class FinBERTHybrid(nn.Module):
    def __init__(self, n_classes=3, n_lm_feats=5, n_temporal=3):
        super().__init__()
        self.bert      = AutoModel.from_pretrained('ProsusAI/finbert')
        self.attention = nn.Linear(768, 1)
        self.dropout   = nn.Dropout(0.3)

        self.lm_branch = nn.Sequential(
            nn.Linear(n_lm_feats, 64), nn.LayerNorm(64), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(64, 64),         nn.LayerNorm(64), nn.GELU(), nn.Dropout(0.1),
        )
        self.temporal_branch = nn.Sequential(
            nn.Linear(n_temporal, 32), nn.LayerNorm(32), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(32, 16),         nn.LayerNorm(16), nn.GELU(),
        )

        _fuse_dim = 768 + 64 + n_lm_feats + 16
        self.gate        = nn.Linear(_fuse_dim, _fuse_dim)
        self.fusion_norm = nn.LayerNorm(_fuse_dim)
        self.classifier  = nn.Linear(_fuse_dim, n_classes)

    def forward(self, input_ids, attention_mask, lm_features, temporal_features):
        hidden = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        attn_scores  = self.attention(hidden).squeeze(-1)
        attn_scores  = attn_scores.masked_fill(attention_mask == 0, float('-inf'))
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        bert_ctx     = self.dropout((attn_weights * hidden).sum(dim=1))

        lm_repr   = self.lm_branch(lm_features)
        lm_skip   = lm_features
        temp_repr = self.temporal_branch(temporal_features)

        fused = torch.cat([bert_ctx, lm_repr, lm_skip, temp_repr], dim=1)
        gate  = torch.sigmoid(self.gate(fused))
        fused = self.fusion_norm(fused * gate)
        return self.classifier(fused)

tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')
model = FinBERTHybrid(n_classes=3, n_lm_feats=5, n_temporal=3).to(device)

CHECKPOINT_PATH = '/content/drive/MyDrive/best_model.pt'  # <-- update to your exact Drive path
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.eval()
print("✅ Checkpoint loaded — no retraining performed")


## Inference function

In [ ]:
LABELS = ['Dovish', 'Neutral', 'Hawkish']  # matches label_map 0/1/2

def classify_tone(text, temporal_override=None):
    """
    text: any headline/speech snippet
    temporal_override: optional dict {'yield_change':..,'volatility':..,'Bond_Yield':..}
                        if None, uses the most recent known market row (proxy for 'live')
    """
    lm_raw = compute_lm_features(text)
    lm_vec = np.array([[lm_raw[c] for c in LM_FEATURE_COLS]], dtype=np.float32)
    lm_scaled = lm_scaler.transform(lm_vec)

    if temporal_override is None:
        temp_raw = df[TEMPORAL_COLS].iloc[-1].values.reshape(1, -1).astype(np.float32)
    else:
        temp_raw = np.array([[temporal_override[c] for c in TEMPORAL_COLS]], dtype=np.float32)
    temp_scaled = temporal_scaler.transform(temp_raw)

    enc = tokenizer(text, truncation=True, max_length=512, padding='max_length', return_tensors='pt')

    with torch.no_grad():
        logits = model(
            enc['input_ids'].to(device),
            enc['attention_mask'].to(device),
            torch.tensor(lm_scaled, dtype=torch.float32).to(device),
            torch.tensor(temp_scaled, dtype=torch.float32).to(device),
        )
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred = int(np.argmax(probs))

    return {
        'tone': LABELS[pred],
        'confidence': float(probs[pred]),
        'probabilities': {LABELS[i]: float(probs[i]) for i in range(3)}
    }

# quick sanity check
print(classify_tone("Inflation risks remain tilted to the upside and policy will stay restrictive."))
print(classify_tone("Growth momentum is fading and downside risks warrant a cautious, supportive stance."))


## Serve as an API for the Trading Floor UI

In [ ]:
!pip install flask flask-ngrok pyngrok -q
from flask import Flask, request, jsonify
from flask_ngrok import run_with_ngrok

app = Flask(__name__)
run_with_ngrok(app)

@app.route('/classify', methods=['POST'])
def classify():
    data = request.get_json(force=True)
    text = data.get('text', '')
    temporal_override = data.get('temporal', None)
    result = classify_tone(text, temporal_override)
    return jsonify(result)

app.run()
# Running this cell prints a public *.ngrok-free.app URL — use that as the API base
# in the Trading Floor frontend's fetch() call to /classify.
